<a href="https://colab.research.google.com/github/David2204269/RAHCE/blob/v2f/notebooks/v2e_HMDB51_4_DATASETS_H5_MP4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAHCE — HMDB51 → V2E — 4 DATASETS H5 + MP4
Esta versión genera únicamente los cuatro datasets asignados:

1. `dvs128_t0.1`
2. `dvs128_t0.2`
3. `dvs128_t0.3`
4. `dvs640_t0.3`

Cada dataset contiene:

- 51 clases
- 5 videos por clase
- 255 videos
- un archivo `.h5` y un `.mp4` por video

Total:

- **4 datasets**
- **1020 conversiones V2E**
- **1020 archivos H5**
- **1020 archivos MP4**

La salida se organiza así:

```text
HMDB51_V2E/
├── dvs128_t0.1/
│   ├── brush_hair/
│   │   ├── video_01.h5
│   │   ├── video_01.mp4
│   │   └── ...
│   └── ... 51 clases
├── dvs128_t0.2/
├── dvs128_t0.3/
├── dvs640_t0.3/
├── selection_manifest.csv
├── processing_manifest.csv
└── progress.json
```

La ejecución es reanudable desde Google Drive. Una conversión se considera completa solamente si existen ambos archivos finales `.h5 + .mp4`.

In [ ]:
# ============================================================
# 1. Preparación del entorno
# ============================================================
import os
import sys
import shutil
import subprocess
from pathlib import Path

print("📦 Instalando utilidades del sistema...")

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=False,
)

install = subprocess.run(
    [
        "apt-get", "install", "-y", "-qq",
        "ffmpeg",
        "unrar",
        "p7zip-full",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

if install.returncode != 0:
    print("⚠️ Salida de apt:")
    print("\n".join(install.stdout.splitlines()[-30:]))

UNRAR_EXECUTABLE = shutil.which("unrar")

if UNRAR_EXECUTABLE is None:
    raise RuntimeError(
        "No se encontró 'unrar'. "
        "No se continuará con la extracción del HMDB51."
    )

print("✅ Extractor RAR:", UNRAR_EXECUTABLE)

packages = [
    "numba",
    "engineering-notation",
    "opencv-contrib-python",
    "argcomplete",
    "dv-processing",
    "tqdm",
    "h5py",
    "psutil",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *packages,
    ]
)

if shutil.which("nvidia-smi"):
    subprocess.run(
        ["nvidia-smi"],
        check=False,
    )
else:
    print(
        "ℹ️ GPU NVIDIA no detectada. "
        "V2E puede funcionar con DISABLE_SLOMO=True."
    )


📦 Instalando utilidades del sistema...
✅ Extractor RAR: /usr/bin/unrar
ℹ️ GPU NVIDIA no detectada. V2E puede funcionar con DISABLE_SLOMO=True.


In [ ]:
# ============================================================
# 2. Instalar V2E + compatibilidad NumPy
# ============================================================
V2E_DIR = Path("/content/v2e")

if not V2E_DIR.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/SensorsINI/v2e", str(V2E_DIR)
    ])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-e", str(V2E_DIR)
])

v2e_text_writer = V2E_DIR / "v2ecore" / "output" / "ae_text_output.py"
if v2e_text_writer.exists():
    content = v2e_text_writer.read_text(encoding="utf-8")
    patched = content.replace("astype(np.float)", "astype(float)")
    if patched != content:
        v2e_text_writer.write_text(patched, encoding="utf-8")
        print("✅ Parche NumPy aplicado: np.float → float")
    else:
        print("✅ Compatibilidad NumPy verificada.")

V2E_EXECUTABLE = shutil.which("v2e")
if not V2E_EXECUTABLE:
    raise RuntimeError("No se encontró el ejecutable v2e.")
print("✅ V2E:", V2E_EXECUTABLE)


✅ Parche NumPy aplicado: np.float → float
✅ V2E: /usr/local/bin/v2e


In [ ]:
# ============================================================
# 3. Montar Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 4. Configuración

Modifica principalmente:

- `DATASET_RAR_PATH`: ruta del `.rar` original.
- `FINAL_OUTPUT_ROOT`: carpeta donde se crearán los **6 datasets**.

Los seis datasets se crean automáticamente dentro de `FINAL_OUTPUT_ROOT`.

In [ ]:
# ============================================================
# 4. CONFIGURACIÓN
# ============================================================
import os
from pathlib import Path

DATASET_RAR_PATH = "/content/drive/MyDrive/hmdb51.rar"  #@param {type:"string"}
FINAL_OUTPUT_ROOT = "/content/drive/MyDrive/DATASET_GEN"  #@param {type:"string"}

LOCAL_ROOT = Path("/content/RAHCE_HMDB51_V2E")
LOCAL_ARCHIVE_DIR = LOCAL_ROOT / "archive"
LOCAL_SELECTED_DATASET = LOCAL_ROOT / "selected_5_per_class"
LOCAL_WORK_DIR = LOCAL_ROOT / "work"
LOCAL_NESTED_RARS = LOCAL_ROOT / "nested_rars"
LOCAL_FULL_EXTRACT = LOCAL_ROOT / "full_extract"

EXPECTED_CLASSES = 51
VIDEOS_PER_CLASS = 5
SELECTION_SEED = 2026
VIDEO_EXTENSIONS = {".avi", ".mp4", ".mov", ".mkv", ".wmv", ".m4v"}
REBUILD_SELECTION = False

# ÚNICOS parámetros del barrido
THRES_VALUES = [0.10, 0.20, 0.30]
OUTPUT_MODES = ["dvs128", "dvs640"]

DATASET_CONFIGS = [
    ("dvs128", 0.10),
    ("dvs128", 0.20),
    ("dvs128", 0.30),
    ("dvs640", 0.30),
]

THRESHOLDS_BY_CAMERA = {
    "dvs128": [0.10, 0.20, 0.30],
    "dvs640": [0.30],
}

def dataset_name(camera, threshold):
    return f"{camera}_t{threshold:.1f}"

# Parámetros fijos
CUTOFF_HZ = 300
DISABLE_SLOMO = True
SIGMA_THRES = 0.03
LEAK_RATE_HZ = 0.1
SHOT_NOISE_RATE_HZ = 0.001
DVS_EXPOSURE = "duration 0.01"
V2E_LOG_LEVEL = "ERROR"

# No forzar valores que no estaban definidos originalmente.
INPUT_FRAME_RATE = None
INPUT_SLOWMOTION_FACTOR = None

# Rendimiento
def available_cpu_threads():
    try:
        return len(os.sched_getaffinity(0))
    except Exception:
        return os.cpu_count() or 1

CPU_THREADS = available_cpu_threads()
REQUESTED_PARALLEL_V2E = 2
MAX_PARALLEL_V2E = max(1, min(REQUESTED_PARALLEL_V2E, CPU_THREADS))
V2E_THREADS_PER_JOB = max(1, CPU_THREADS // MAX_PARALLEL_V2E)
USE_GPU_FFMPEG = True
SKIP_COMPLETED = True
DELETE_TEMP_AFTER_SUCCESS = True
MIN_LOCAL_FREE_GB = 8

FINAL_OUTPUT_ROOT = Path(FINAL_OUTPUT_ROOT)
FINAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROGRESS_JSON = FINAL_OUTPUT_ROOT / "progress.json"
for p in [LOCAL_ROOT, LOCAL_ARCHIVE_DIR, LOCAL_SELECTED_DATASET, LOCAL_WORK_DIR, LOCAL_NESTED_RARS, LOCAL_FULL_EXTRACT]:
    p.mkdir(parents=True, exist_ok=True)

print("🔧 Configuración activa:")
print("   clases:", EXPECTED_CLASSES)
print("   videos/clase:", VIDEOS_PER_CLASS)
print("   videos objetivo:", EXPECTED_CLASSES * VIDEOS_PER_CLASS)
print("   cámaras:", OUTPUT_MODES)
print("   thresholds:", THRES_VALUES)
print("   cutoff:", CUTOFF_HZ, "Hz")
print("   datasets/configuraciones por video:", len(DATASET_CONFIGS))
print("   conversiones objetivo:", EXPECTED_CLASSES * VIDEOS_PER_CLASS * len(DATASET_CONFIGS))
print("   CPU threads:", CPU_THREADS)
print("   V2E paralelos:", MAX_PARALLEL_V2E)
print("   hilos/V2E:", V2E_THREADS_PER_JOB)


🔧 Configuración activa:
   clases: 51
   videos/clase: 5
   videos objetivo: 255
   cámaras: ['dvs128', 'dvs640']
   thresholds: [0.1, 0.2, 0.3]
   cutoff: 300 Hz
   datasets/configuraciones por video: 4
   conversiones objetivo: 1020
   CPU threads: 2
   V2E paralelos: 2
   hilos/V2E: 1


In [ ]:
import csv
import random
import zlib
import shutil
import subprocess
from pathlib import Path

SELECTION_MANIFEST = FINAL_OUTPUT_ROOT / "selection_manifest.csv"


def stable_class_rng(class_name):
    class_seed = (
        SELECTION_SEED
        + (zlib.crc32(class_name.encode("utf-8")) & 0xffffffff)
    )
    return random.Random(class_seed)


def choose_five_paths(class_name, video_paths):
    video_paths = sorted(video_paths, key=lambda p: p.name.lower())

    if len(video_paths) < VIDEOS_PER_CLASS:
        raise RuntimeError(
            f"La clase '{class_name}' solo contiene "
            f"{len(video_paths)} videos; se necesitan "
            f"{VIDEOS_PER_CLASS}."
        )

    rng = stable_class_rng(class_name)
    return sorted(
        rng.sample(video_paths, VIDEOS_PER_CLASS),
        key=lambda p: p.name.lower()
    )


def copy_archive_local(src_path):
    src = Path(src_path)

    if not src.is_file():
        raise FileNotFoundError(f"No existe el dataset RAR: {src}")

    dst = LOCAL_ARCHIVE_DIR / src.name

    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        print("📥 Copiando RAR desde Drive al SSD local...")
        shutil.copy2(src, dst)
    else:
        print("✅ RAR ya disponible localmente.")

    return dst




def extract_archive_full(archive_path, output_dir):
    """
    Extrae un RAR usando el ejecutable oficial `unrar`.

    Se usa `unrar` porque las pruebas anteriores con 7z y unar
    no fueron fiables con este archivo HMDB51.
    """
    archive_path = Path(archive_path)
    output_dir = Path(output_dir)

    shutil.rmtree(output_dir, ignore_errors=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    unrar = shutil.which("unrar")

    if unrar is None:
        raise RuntimeError(
            "No se encontró 'unrar'. "
            "Ejecuta nuevamente la celda 1."
        )

    print(f"📦 Extrayendo con unrar: {archive_path.name}")

    result = subprocess.run(
        [
            unrar,
            "x",
            "-o+",
            "-idq",
            str(archive_path),
            str(output_dir) + "/",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        tail = "\\n".join(
            result.stdout.splitlines()[-40:]
        )

        raise RuntimeError(
            "unrar no pudo extraer el archivo.\\n"
            f"Código: {result.returncode}\\n"
            f"Últimas líneas:\\n{tail}"
        )

    videos = [
        p for p in output_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in VIDEO_EXTENSIONS
    ]

    nested_rars = [
        p for p in output_dir.rglob("*.rar")
        if p.is_file()
    ]

    if not videos and not nested_rars:
        raise RuntimeError(
            "La extracción terminó sin error, "
            "pero no se encontraron videos ni RAR internos."
        )

    print(
        f"✅ Extracción terminada | "
        f"videos detectados={len(videos):,} | "
        f"RAR internos={len(nested_rars)}"
    )

    return output_dir


def find_video_files(root):
    root = Path(root)
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    )


def find_rar_files(root):
    root = Path(root)
    return sorted(
        p for p in root.rglob("*.rar")
        if p.is_file()
    )


def selected_dataset_is_valid():
    if not LOCAL_SELECTED_DATASET.exists():
        return False

    classes = sorted(
        p for p in LOCAL_SELECTED_DATASET.iterdir()
        if p.is_dir()
    )

    if len(classes) != EXPECTED_CLASSES:
        return False

    for class_dir in classes:
        videos = [
            v for v in class_dir.iterdir()
            if v.is_file() and v.suffix.lower() in VIDEO_EXTENSIONS
        ]
        if len(videos) != VIDEOS_PER_CLASS:
            return False

    return True


def copy_selected_videos(class_name, selected, rows):
    class_dir = LOCAL_SELECTED_DATASET / class_name
    class_dir.mkdir(parents=True, exist_ok=True)

    for src in selected:
        dst = class_dir / src.name
        shutil.copy2(src, dst)
        rows.append({
            "class": class_name,
            "selected_video": src.name,
            "source_path": str(src),
            "local_path": str(dst),
        })


def write_selection_manifest(rows):
    SELECTION_MANIFEST.parent.mkdir(parents=True, exist_ok=True)

    with SELECTION_MANIFEST.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "class",
                "selected_video",
                "source_path",
                "local_path",
            ],
        )
        w.writeheader()
        w.writerows(rows)

    print("📄 Manifest de selección:", SELECTION_MANIFEST)


def prepare_from_direct_class_folders(extracted_root, rows):
    extracted_root = Path(extracted_root)
    candidate_dirs = []

    for d in extracted_root.rglob("*"):
        if not d.is_dir():
            continue

        direct_videos = [
            p for p in d.iterdir()
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        ]

        if direct_videos:
            candidate_dirs.append((d, direct_videos))

    candidate_dirs = sorted(
        candidate_dirs,
        key=lambda item: str(item[0]).lower()
    )

    if len(candidate_dirs) != EXPECTED_CLASSES:
        return False

    print(f"📁 Detectadas {len(candidate_dirs)} carpetas de clase.")

    for idx, (class_dir, videos) in enumerate(candidate_dirs, 1):
        class_name = class_dir.name
        selected = choose_five_paths(class_name, videos)
        copy_selected_videos(class_name, selected, rows)

        print(
            f"✅ [{idx}/{len(candidate_dirs)}] "
            f"{class_name}: {VIDEOS_PER_CLASS}/{len(videos)} videos"
        )

    return True


def prepare_from_nested_class_rars(extracted_root, rows):
    class_rars = find_rar_files(extracted_root)

    if not class_rars:
        return False

    if len(class_rars) != EXPECTED_CLASSES:
        print(
            f"⚠️ Se encontraron {len(class_rars)} RAR internos; "
            f"se esperaban {EXPECTED_CLASSES}."
        )
        return False

    print(f"📦 Detectados {len(class_rars)} RAR internos de clase.")

    shutil.rmtree(LOCAL_NESTED_RARS, ignore_errors=True)
    LOCAL_NESTED_RARS.mkdir(parents=True, exist_ok=True)

    for idx, class_rar in enumerate(sorted(class_rars), 1):
        class_name = class_rar.stem
        class_extract = LOCAL_NESTED_RARS / class_name

        extract_archive_full(class_rar, class_extract)

        videos = find_video_files(class_extract)
        selected = choose_five_paths(class_name, videos)
        copy_selected_videos(class_name, selected, rows)

        shutil.rmtree(class_extract, ignore_errors=True)

        print(
            f"✅ [{idx}/{len(class_rars)}] "
            f"{class_name}: {VIDEOS_PER_CLASS}/{len(videos)} videos"
        )

    return True


def prepare_hmdb51_sample(rar_path):
    if not REBUILD_SELECTION and selected_dataset_is_valid():
        print("✅ El subconjunto 51×5 ya existe localmente; se reutiliza.")
        return LOCAL_SELECTED_DATASET

    shutil.rmtree(LOCAL_SELECTED_DATASET, ignore_errors=True)
    LOCAL_SELECTED_DATASET.mkdir(parents=True, exist_ok=True)

    # Ensure local archive is also cleared for a fresh start if REBUILD_SELECTION is True
    if REBUILD_SELECTION:
        print("🗑️ Forzando re-copia del RAR, eliminando archivo local existente.")
        shutil.rmtree(LOCAL_ARCHIVE_DIR, ignore_errors=True)
        LOCAL_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True) # Recreate the directory

    local_rar = copy_archive_local(rar_path)
    extract_archive_full(local_rar, LOCAL_FULL_EXTRACT)

    rows = []

    try:
        ok = prepare_from_direct_class_folders(
            LOCAL_FULL_EXTRACT,
            rows
        )

        if not ok:
            rows.clear()
            shutil.rmtree(LOCAL_SELECTED_DATASET, ignore_errors=True)
            LOCAL_SELECTED_DATASET.mkdir(parents=True, exist_ok=True)

            ok = prepare_from_nested_class_rars(
                LOCAL_FULL_EXTRACT,
                rows
            )

        if not ok:
            raise RuntimeError(
                "No se pudo identificar una estructura válida de HMDB51. "
                "Se esperaban 51 carpetas de clase con videos "
                "o 51 RAR internos de clase."
            )

        classes = sorted(
            p for p in LOCAL_SELECTED_DATASET.iterdir()
            if p.is_dir()
        )

        total_videos = sum(
            len([
                v for v in c.iterdir()
                if v.is_file() and v.suffix.lower() in VIDEO_EXTENSIONS
            ])
            for c in classes
        )

        print("\n" + "=" * 72)
        print("📊 SELECCIÓN DEL DATASET")
        print("   clases:", len(classes))
        print("   videos:", total_videos)
        print("   esperado:", EXPECTED_CLASSES * VIDEOS_PER_CLASS)
        print("=" * 72)

        if len(classes) != EXPECTED_CLASSES:
            raise RuntimeError(
                f"Se esperaban {EXPECTED_CLASSES} clases y hay {len(classes)}."
            )

        if total_videos != EXPECTED_CLASSES * VIDEOS_PER_CLASS:
            raise RuntimeError(
                "La selección no contiene exactamente 5 videos por clase."
            )

        write_selection_manifest(rows)

    finally:
        print("🧹 Eliminando extracción completa del dataset...")
        shutil.rmtree(LOCAL_FULL_EXTRACT, ignore_errors=True)
        shutil.rmtree(LOCAL_NESTED_RARS, ignore_errors=True)

    return LOCAL_SELECTED_DATASET


## 6. Preparar el subconjunto 51 × 5

La preparación usa `unrar` y después selecciona de forma reproducible exactamente:

```text
51 clases × 5 videos = 255 videos
```

La selección usa una semilla fija, por lo que al reiniciar el runtime se vuelven a seleccionar los mismos videos.

Todavía no se ejecuta V2E en esta etapa.

In [ ]:
# ============================================================
# 6. SELECCIONAR 5 VIDEOS POR CLASE
# ============================================================
selected_dataset_root = prepare_hmdb51_sample(DATASET_RAR_PATH)


📥 Copiando RAR desde Drive al SSD local...
📦 Extrayendo con unrar: hmdb51.rar
✅ Extracción terminada | videos detectados=6,766 | RAR internos=0
📁 Detectadas 51 carpetas de clase.
✅ [1/51] brush_hair: 5/107 videos
✅ [2/51] cartwheel: 5/107 videos
✅ [3/51] catch: 5/102 videos
✅ [4/51] chew: 5/109 videos
✅ [5/51] clap: 5/130 videos
✅ [6/51] climb: 5/108 videos
✅ [7/51] climb_stairs: 5/112 videos
✅ [8/51] dive: 5/127 videos
✅ [9/51] draw_sword: 5/103 videos
✅ [10/51] dribble: 5/145 videos
✅ [11/51] drink: 5/164 videos
✅ [12/51] eat: 5/108 videos
✅ [13/51] fall_floor: 5/136 videos
✅ [14/51] fencing: 5/116 videos
✅ [15/51] flic_flac: 5/107 videos
✅ [16/51] golf: 5/105 videos
✅ [17/51] handstand: 5/113 videos
✅ [18/51] hit: 5/127 videos
✅ [19/51] hug: 5/118 videos
✅ [20/51] jump: 5/151 videos
✅ [21/51] kick: 5/130 videos
✅ [22/51] kick_ball: 5/128 videos
✅ [23/51] kiss: 5/102 videos
✅ [24/51] laugh: 5/128 videos
✅ [25/51] pick: 5/106 videos
✅ [26/51] pour: 5/106 videos
✅ [27/51] pullup: 5/104

In [ ]:
# ============================================================
# 7. Diagnóstico antes de V2E
# ============================================================
import shutil

selected_classes = sorted(
    p for p in LOCAL_SELECTED_DATASET.iterdir()
    if p.is_dir()
)

selected_videos = sorted(
    v
    for c in selected_classes
    for v in c.iterdir()
    if v.is_file() and v.suffix.lower() in VIDEO_EXTENSIONS
)

free_gb = shutil.disk_usage("/content").free / 1024**3

print("🔎 Diagnóstico final:")
print("   clases seleccionadas:", len(selected_classes))
print("   videos seleccionados:", len(selected_videos))
print("   videos por clase:", VIDEOS_PER_CLASS)
print("   datasets independientes:", len(DATASET_CONFIGS))
print("   configuraciones:")

for camera, threshold in DATASET_CONFIGS:
    print(f"      - {dataset_name(camera, threshold)}")

print("   videos por dataset:", len(selected_videos))
print(
    "   conversiones V2E totales:",
    len(selected_videos) * len(DATASET_CONFIGS)
)
print(
    "   archivos finales H5:",
    len(selected_videos) * len(DATASET_CONFIGS)
)
print(
    "   archivos finales MP4:",
    len(selected_videos) * len(DATASET_CONFIGS)
)
print("   CPU threads:", CPU_THREADS)
print("   V2E paralelos:", MAX_PARALLEL_V2E)
print("   hilos por V2E:", V2E_THREADS_PER_JOB)
print(f"   espacio libre local: {free_gb:.1f} GB")
print("   raíz de salida:", FINAL_OUTPUT_ROOT)

if len(selected_classes) != EXPECTED_CLASSES:
    raise RuntimeError(
        f"Se esperaban {EXPECTED_CLASSES} clases y hay {len(selected_classes)}."
    )

if len(selected_videos) != EXPECTED_CLASSES * VIDEOS_PER_CLASS:
    raise RuntimeError(
        "La selección debe contener exactamente "
        f"{EXPECTED_CLASSES * VIDEOS_PER_CLASS} videos."
    )

if free_gb < MIN_LOCAL_FREE_GB:
    raise RuntimeError(
        f"Espacio local insuficiente: {free_gb:.1f} GB libres."
    )


🔎 Diagnóstico final:
   clases seleccionadas: 51
   videos seleccionados: 255
   videos por clase: 5
   datasets independientes: 4
   configuraciones:
      - dvs128_t0.1
      - dvs128_t0.2
      - dvs128_t0.3
      - dvs640_t0.3
   videos por dataset: 255
   conversiones V2E totales: 1020
   archivos finales H5: 1020
   archivos finales MP4: 1020
   CPU threads: 2
   V2E paralelos: 2
   hilos por V2E: 1
   espacio libre local: 83.4 GB
   raíz de salida: /content/drive/MyDrive/DATASET_GEN


In [ ]:
# ============================================================
# 8. Pipeline V2E H5 + MP4 + checkpoint durable
# ============================================================
import os
import csv
import gc
import json
import time
import shutil
import subprocess
from dataclasses import dataclass
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import h5py

PROCESSING_MANIFEST = FINAL_OUTPUT_ROOT / "processing_manifest.csv"
PROCESSING_FIELDS = [
    "timestamp",
    "class",
    "source_video",
    "dataset",
    "camera",
    "threshold",
    "cutoff_hz",
    "event_count",
    "v2e_seconds",
    "mp4_seconds",
    "total_config_seconds",
    "h5_path",
    "mp4_path",
]


def format_duration(seconds):
    seconds = max(0, int(round(seconds)))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def ffmpeg_has_nvenc():
    if not USE_GPU_FFMPEG:
        return False

    r = subprocess.run(
        ["ffmpeg", "-hide_banner", "-encoders"],
        capture_output=True,
        text=True,
        check=False,
    )
    return "h264_nvenc" in (r.stdout + r.stderr)


def v2e_env():
    env = os.environ.copy()
    n = str(V2E_THREADS_PER_JOB)

    for key in [
        "OMP_NUM_THREADS",
        "OPENBLAS_NUM_THREADS",
        "MKL_NUM_THREADS",
        "NUMEXPR_NUM_THREADS",
    ]:
        env[key] = n

    env["PYTHONUNBUFFERED"] = "1"
    return env


# ------------------------------------------------------------
# Organización:
# FINAL_OUTPUT_ROOT/
#   dvs128_t0.1/
#       clase/
#           video.h5
#           video.mp4
#   ...
# ------------------------------------------------------------
def final_paths(class_name, video_basename, camera, threshold):
    dataset_root = FINAL_OUTPUT_ROOT / dataset_name(camera, threshold)
    class_dir = dataset_root / class_name

    h5_path = class_dir / f"{video_basename}.h5"
    mp4_path = class_dir / f"{video_basename}.mp4"

    return dataset_root, class_dir, h5_path, mp4_path


def complete(class_name, video_basename, camera, threshold):
    _, _, h5_path, mp4_path = final_paths(
        class_name,
        video_basename,
        camera,
        threshold,
    )

    return (
        h5_path.is_file()
        and h5_path.stat().st_size > 0
        and mp4_path.is_file()
        and mp4_path.stat().st_size > 0
    )


def safe_copy(src, dst):
    """
    Copia atómica a Drive.

    Si Colab se interrumpe durante la copia, queda un archivo .part
    que NO se considera una salida final válida.
    """
    src = Path(src)
    dst = Path(dst)

    dst.parent.mkdir(parents=True, exist_ok=True)
    temp = dst.with_name(dst.name + ".part")

    if temp.exists():
        temp.unlink()

    shutil.copy2(src, temp)
    os.replace(temp, dst)


def cleanup_part_files():
    removed = 0

    if FINAL_OUTPUT_ROOT.exists():
        for p in FINAL_OUTPUT_ROOT.rglob("*.part"):
            try:
                p.unlink()
                removed += 1
            except OSError:
                pass

    if removed:
        print(
            f"🧹 Eliminados {removed} archivo(s) .part "
            "de una ejecución interrumpida."
        )


def config_key(class_name, video_basename, camera, threshold):
    return (
        f"{dataset_name(camera, threshold)}/"
        f"{class_name}/{video_basename}"
    )


def atomic_write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_name(path.name + ".part")

    with tmp.open("w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )

    os.replace(tmp, path)


def save_progress_snapshot(dataset_items=None, last_completed=None):
    completed_keys = []

    if dataset_items is not None:
        for class_name, video_path in dataset_items:
            video_basename = Path(video_path).stem

            for camera, threshold in DATASET_CONFIGS:
                if complete(
                    class_name,
                    video_basename,
                    camera,
                    threshold,
                ):
                    completed_keys.append(
                        config_key(
                            class_name,
                            video_basename,
                            camera,
                            threshold,
                        )
                    )

    total = (
        len(dataset_items) * len(DATASET_CONFIGS)
        if dataset_items is not None
        else None
    )

    data = {
        "updated_at": datetime.now().isoformat(timespec="seconds"),
        "datasets": [
            dataset_name(camera, threshold)
            for camera, threshold in DATASET_CONFIGS
        ],
        "videos_per_dataset": (
            len(dataset_items) if dataset_items is not None else None
        ),
        "total_configurations": total,
        "completed_configurations": len(completed_keys),
        "pending_configurations": (
            total - len(completed_keys)
            if total is not None
            else None
        ),
        "last_completed": last_completed,
        "completed": completed_keys,
    }

    atomic_write_json(PROGRESS_JSON, data)
    return data


def append_manifest(row):
    new_file = not PROCESSING_MANIFEST.exists()

    with PROCESSING_MANIFEST.open(
        "a",
        newline="",
        encoding="utf-8",
    ) as f:
        w = csv.DictWriter(
            f,
            fieldnames=PROCESSING_FIELDS,
        )

        if new_file:
            w.writeheader()

        w.writerow(row)


def find_dvs_avi(folder):
    folder = Path(folder)

    preferred = folder / "dvs-video.avi"

    if preferred.is_file() and preferred.stat().st_size > 0:
        return preferred

    candidates = sorted(folder.glob("*dvs*.avi"))

    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        f"No se encontró AVI DVS en {folder}"
    )


def validate_h5(h5_path):
    """
    Verifica que el HDF5 tenga el dataset 'events'
    con cuatro columnas generadas por V2E.
    """
    h5_path = Path(h5_path)

    if not h5_path.is_file() or h5_path.stat().st_size == 0:
        raise RuntimeError("H5 inexistente o vacío.")

    with h5py.File(h5_path, "r") as h5:
        if "events" not in h5:
            raise RuntimeError(
                "El H5 no contiene el dataset 'events'."
            )

        ds = h5["events"]

        if ds.ndim != 2 or ds.shape[1] != 4:
            raise RuntimeError(
                f"Formato H5 inesperado: events.shape={ds.shape}"
            )

        return int(ds.shape[0])


@dataclass
class V2EResult:
    class_name: str
    video_basename: str
    camera: str
    threshold: float
    local_dir: Path
    h5_path: Path
    avi_path: Path
    event_count: int
    v2e_seconds: float


def build_v2e_command(
    video_path,
    local_dir,
    threshold,
    camera,
):
    cmd = [
        V2E_EXECUTABLE,
        "-i", str(video_path),
        "-o", str(local_dir),
        "--overwrite",

        # HDF5 de eventos: salida de datos definitiva
        "--dvs_h5", "events.h5",

        # No generar TXT ni AEDAT2
        "--dvs_text", "None",
        "--dvs_aedat2", "None",

        "--no_preview",
        "--dvs_exposure", *DVS_EXPOSURE.split(),

        "--pos_thres", str(threshold),
        "--neg_thres", str(threshold),
        "--sigma_thres", str(SIGMA_THRES),
        "--cutoff_hz", str(CUTOFF_HZ),
        "--leak_rate_hz", str(LEAK_RATE_HZ),
        "--shot_noise_rate_hz", str(SHOT_NOISE_RATE_HZ),

        f"--{camera}",
        "--log_level", V2E_LOG_LEVEL,
    ]

    if INPUT_FRAME_RATE is not None:
        cmd += [
            "--input_frame_rate",
            str(INPUT_FRAME_RATE),
        ]

    if INPUT_SLOWMOTION_FACTOR is not None:
        cmd += [
            "--input_slowmotion_factor",
            str(INPUT_SLOWMOTION_FACTOR),
        ]

    if DISABLE_SLOMO:
        cmd += [
            "--disable_slomo",
            "--auto_timestamp_resolution",
            "false",
        ]

    return cmd


def run_v2e_only(
    video_path,
    class_name,
    camera,
    threshold,
):
    video_path = Path(video_path)
    video_basename = video_path.stem

    local_dir = (
        LOCAL_WORK_DIR
        / dataset_name(camera, threshold)
        / class_name
        / video_basename
    )

    shutil.rmtree(
        local_dir,
        ignore_errors=True,
    )
    local_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    cmd = build_v2e_command(
        video_path,
        local_dir,
        threshold,
        camera,
    )

    print(
        f"▶️ V2E | {dataset_name(camera, threshold)} | "
        f"{class_name}/{video_path.name}"
    )

    t0 = time.perf_counter()

    result = subprocess.run(
        cmd,
        env=v2e_env(),
        check=False,
    )

    elapsed = time.perf_counter() - t0

    if result.returncode != 0:
        raise RuntimeError(
            f"V2E terminó con código {result.returncode}"
        )

    h5_path = local_dir / "events.h5"

    if not h5_path.is_file():
        candidates = sorted(local_dir.glob("*.h5"))

        if not candidates:
            raise FileNotFoundError(
                "V2E no generó HDF5."
            )

        h5_path = candidates[0]

    event_count = validate_h5(h5_path)
    avi_path = find_dvs_avi(local_dir)

    print(
        f"✅ V2E | {dataset_name(camera, threshold)} | "
        f"{event_count:,} eventos | "
        f"{format_duration(elapsed)}"
    )

    return V2EResult(
        class_name=class_name,
        video_basename=video_basename,
        camera=camera,
        threshold=threshold,
        local_dir=local_dir,
        h5_path=h5_path,
        avi_path=avi_path,
        event_count=event_count,
        v2e_seconds=elapsed,
    )


def avi_to_mp4(avi_path, mp4_path):
    avi_path = Path(avi_path)
    mp4_path = Path(mp4_path)

    t0 = time.perf_counter()

    base = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        "-y",
        "-i", str(avi_path),
        "-an",
    ]

    ok = False

    if ffmpeg_has_nvenc():
        print("🚀 MP4: h264_nvenc")

        cmd = base + [
            "-c:v", "h264_nvenc",
            "-preset", "p4",
            "-cq", "23",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(mp4_path),
        ]

        r = subprocess.run(
            cmd,
            check=False,
        )

        ok = (
            r.returncode == 0
            and mp4_path.is_file()
            and mp4_path.stat().st_size > 0
        )

    if not ok:
        print("ℹ️ MP4: libx264")

        if mp4_path.exists():
            mp4_path.unlink()

        cmd = base + [
            "-c:v", "libx264",
            "-crf", "23",
            "-preset", "veryfast",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(mp4_path),
        ]

        subprocess.run(
            cmd,
            check=True,
        )

    elapsed = time.perf_counter() - t0

    if (
        not mp4_path.is_file()
        or mp4_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "No se generó un MP4 válido."
        )

    print(
        "✅ MP4 |",
        format_duration(elapsed),
    )

    return elapsed


def finalize_result(
    source_video,
    result,
    dataset_items=None,
):
    dataset_root, class_dir, final_h5, final_mp4 = final_paths(
        result.class_name,
        result.video_basename,
        result.camera,
        result.threshold,
    )

    class_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    local_mp4 = (
        result.local_dir
        / f"{result.video_basename}.mp4"
    )

    mp4_seconds = avi_to_mp4(
        result.avi_path,
        local_mp4,
    )

    # Validar H5 ANTES de copiar.
    event_count = validate_h5(
        result.h5_path
    )

    print("📤 Copiando H5 + MP4 finales a Drive...")

    safe_copy(
        result.h5_path,
        final_h5,
    )

    safe_copy(
        local_mp4,
        final_mp4,
    )

    # Validar otra vez el H5 ya guardado en Drive.
    copied_event_count = validate_h5(
        final_h5
    )

    if copied_event_count != event_count:
        raise RuntimeError(
            "El número de eventos cambió después de copiar el H5."
        )

    if not complete(
        result.class_name,
        result.video_basename,
        result.camera,
        result.threshold,
    ):
        raise RuntimeError(
            "La salida final H5 + MP4 no superó la validación."
        )

    total_seconds = (
        result.v2e_seconds
        + mp4_seconds
    )

    dset = dataset_name(
        result.camera,
        result.threshold,
    )

    append_manifest({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "class": result.class_name,
        "source_video": str(source_video),
        "dataset": dset,
        "camera": result.camera,
        "threshold": result.threshold,
        "cutoff_hz": CUTOFF_HZ,
        "event_count": event_count,
        "v2e_seconds": round(result.v2e_seconds, 3),
        "mp4_seconds": round(mp4_seconds, 3),
        "total_config_seconds": round(total_seconds, 3),
        "h5_path": str(final_h5),
        "mp4_path": str(final_mp4),
    })

    last_key = config_key(
        result.class_name,
        result.video_basename,
        result.camera,
        result.threshold,
    )

    if dataset_items is not None:
        snapshot = save_progress_snapshot(
            dataset_items=dataset_items,
            last_completed=last_key,
        )

        print(
            f"💾 CHECKPOINT: "
            f"{snapshot['completed_configurations']}/"
            f"{snapshot['total_configurations']} "
            "pares H5+MP4 completos"
        )

    if DELETE_TEMP_AFTER_SUCCESS:
        shutil.rmtree(
            result.local_dir,
            ignore_errors=True,
        )

    return total_seconds


In [ ]:
# ============================================================
# 9. Procesamiento por video + reanudación
# ============================================================
def check_local_space():
    free_gb = shutil.disk_usage("/content").free / 1024**3

    if free_gb < MIN_LOCAL_FREE_GB:
        raise RuntimeError(
            f"Espacio local bajo: {free_gb:.1f} GB"
        )

    return free_gb


def process_camera_batch(
    video_path,
    class_name,
    camera,
    thresholds,
    dataset_items,
):
    video_path = Path(video_path)
    video_basename = video_path.stem

    pending = [
        threshold
        for threshold in thresholds
        if not (
            SKIP_COMPLETED
            and complete(
                class_name,
                video_basename,
                camera,
                threshold,
            )
        )
    ]

    completed_here = len(thresholds) - len(pending)

    if completed_here:
        print(
            f"⏭️ {class_name}/{video_path.name} | "
            f"{camera}: "
            f"{completed_here}/{len(thresholds)} "
            "configuraciones ya guardadas"
        )

    if not pending:
        return

    check_local_space()

    print("\n" + "=" * 72)
    print(
        f"📷 {class_name}/{video_path.name} | "
        f"cámara={camera}"
    )
    print(
        f"🚀 Pendientes: {len(pending)} thresholds | "
        f"hasta {MAX_PARALLEL_V2E} V2E simultáneos"
    )
    print("=" * 72)

    results = []

    with ThreadPoolExecutor(
        max_workers=min(
            MAX_PARALLEL_V2E,
            len(pending),
        )
    ) as ex:
        futures = {
            ex.submit(
                run_v2e_only,
                video_path,
                class_name,
                camera,
                threshold,
            ): threshold
            for threshold in pending
        }

        for future in as_completed(futures):
            threshold = futures[future]

            try:
                results.append(
                    future.result()
                )

            except Exception as e:
                raise RuntimeError(
                    f"Falló "
                    f"{class_name}/{video_path.name} "
                    f"{camera} t={threshold}: {e}"
                ) from e

    results.sort(
        key=lambda r: r.threshold
    )

    for result in results:
        check_local_space()

        finalize_result(
            video_path,
            result,
            dataset_items=dataset_items,
        )

        gc.collect()


def process_one_video(
    video_path,
    class_name,
    dataset_items,
):
    t0 = time.perf_counter()

    for camera, thresholds in THRESHOLDS_BY_CAMERA.items():
        process_camera_batch(
            video_path,
            class_name,
            camera,
            thresholds,
            dataset_items,
        )

    elapsed = time.perf_counter() - t0

    print(
        f"🎉 Video revisado/terminado: "
        f"{class_name}/{Path(video_path).name} | "
        f"{format_duration(elapsed)}"
    )

    return elapsed


def folder_size_bytes(path):
    total = 0
    path = Path(path)

    if not path.exists():
        return 0

    for root, _, files in os.walk(path):
        for name in files:
            p = Path(root) / name

            try:
                total += p.stat().st_size
            except OSError:
                pass

    return total


## 10. Ejecutar los 4 datasets

Se procesan únicamente:

```text
dvs128_t0.1
dvs128_t0.2
dvs128_t0.3
dvs640_t0.3
```

Cada dataset tendrá los mismos 255 videos seleccionados.

Total:

```text
255 × 4 = 1020 conversiones V2E
```

La salida de cada conversión es:

```text
video.h5
video.mp4
```

Para comenzar el procesamiento completo:

```python
RUN_FULL_DATASET = True
```

Si Colab se desconecta, vuelve a ejecutar las celdas desde arriba. Los pares `.h5 + .mp4` ya terminados en Drive se detectan y se omiten.

In [ ]:
# ============================================================
# 10. EJECUCIÓN COMPLETA / REANUDACIÓN
# ============================================================
RUN_FULL_DATASET = False  #@param {type:"boolean"}

cleanup_part_files()

dataset_items = []

for class_dir in sorted(
    p
    for p in LOCAL_SELECTED_DATASET.iterdir()
    if p.is_dir()
):
    videos = sorted(
        v
        for v in class_dir.iterdir()
        if v.is_file()
        and v.suffix.lower() in VIDEO_EXTENSIONS
    )

    if len(videos) != VIDEOS_PER_CLASS:
        raise RuntimeError(
            f"{class_dir.name} no tiene exactamente "
            f"{VIDEOS_PER_CLASS} videos."
        )

    for video in videos:
        dataset_items.append(
            (class_dir.name, video)
        )

expected = (
    EXPECTED_CLASSES
    * VIDEOS_PER_CLASS
)

if len(dataset_items) != expected:
    raise RuntimeError(
        f"Se esperaban {expected} videos "
        f"y hay {len(dataset_items)}."
    )

resume_snapshot = save_progress_snapshot(
    dataset_items=dataset_items,
    last_completed=None,
)

total_cfg = resume_snapshot[
    "total_configurations"
]
done_cfg = resume_snapshot[
    "completed_configurations"
]
pending_cfg = resume_snapshot[
    "pending_configurations"
]

print("=" * 80)
print("🔄 ESTADO DE LOS 4 DATASETS")
print("   datasets:", len(DATASET_CONFIGS))

for camera, threshold in DATASET_CONFIGS:
    dset = dataset_name(
        camera,
        threshold,
    )

    done_in_dataset = sum(
        complete(
            class_name,
            Path(video_path).stem,
            camera,
            threshold,
        )
        for class_name, video_path in dataset_items
    )

    print(
        f"   {dset}: "
        f"{done_in_dataset}/{len(dataset_items)} videos"
    )

print("-" * 80)
print(
    f"   total pares H5+MP4: "
    f"{done_cfg}/{total_cfg}"
)
print(
    f"   pendientes: {pending_cfg}"
)

if total_cfg:
    print(
        f"   progreso global: "
        f"{100.0 * done_cfg / total_cfg:.2f}%"
    )

print(
    f"   checkpoint: {PROGRESS_JSON}"
)
print("=" * 80)

if not RUN_FULL_DATASET:
    print("ℹ️ Ejecución completa desactivada.")
    print(
        "   Si todo está correcto, cambia "
        "RUN_FULL_DATASET=True."
    )

else:
    if pending_cfg == 0:
        print(
            "🎉 Los 4 datasets están completos."
        )

    else:
        print("=" * 80)
        print(
            "🚀 INICIANDO / REANUDANDO "
            "LOS 4 DATASETS"
        )
        print(
            f"   configuraciones ya listas: "
            f"{done_cfg}"
        )
        print(
            f"   pendientes: "
            f"{pending_cfg}"
        )
        print("=" * 80)

        dataset_start = time.perf_counter()
        failures = []

        for idx, (
            class_name,
            video_path,
        ) in enumerate(
            dataset_items,
            1,
        ):
            video_basename = (
                Path(video_path).stem
            )

            video_is_complete = all(
                complete(
                    class_name,
                    video_basename,
                    camera,
                    threshold,
                )
                for camera, threshold
                in DATASET_CONFIGS
            )

            if video_is_complete:
                print(
                    f"⏭️ [{idx}/{len(dataset_items)}] "
                    f"{class_name}/{video_path.name}: "
                    "4/4 datasets completos"
                )
                continue

            print("\n" + "#" * 80)
            print(
                f"🎬 [{idx}/{len(dataset_items)}] "
                f"{class_name}/{video_path.name}"
            )
            print("#" * 80)

            try:
                process_one_video(
                    video_path,
                    class_name,
                    dataset_items,
                )

            except Exception as e:
                failures.append(
                    (
                        class_name,
                        str(video_path),
                        str(e),
                    )
                )
                print(
                    f"❌ Error: {e}"
                )

            current = save_progress_snapshot(
                dataset_items=dataset_items,
                last_completed=None,
            )

            print(
                f"📊 Progreso durable: "
                f"{current['completed_configurations']}/"
                f"{current['total_configurations']} "
                f"({100.0 * current['completed_configurations'] / current['total_configurations']:.2f}%)"
            )

            print(
                "⏱️ Tiempo de esta sesión:",
                format_duration(
                    time.perf_counter()
                    - dataset_start
                ),
            )

            gc.collect()

        final_snapshot = save_progress_snapshot(
            dataset_items=dataset_items,
            last_completed=None,
        )

        print("\n" + "=" * 80)
        print("📊 SESIÓN FINALIZADA")
        print(
            "   pares H5+MP4 completos:",
            f"{final_snapshot['completed_configurations']}/"
            f"{final_snapshot['total_configurations']}",
        )
        print(
            "   pendientes:",
            final_snapshot[
                "pending_configurations"
            ],
        )
        print(
            "   tiempo de sesión:",
            format_duration(
                time.perf_counter()
                - dataset_start
            ),
        )
        print(
            "   selection manifest:",
            SELECTION_MANIFEST,
        )
        print(
            "   processing manifest:",
            PROCESSING_MANIFEST,
        )
        print(
            "   checkpoint:",
            PROGRESS_JSON,
        )
        print("=" * 80)

        if failures:
            failure_csv = (
                FINAL_OUTPUT_ROOT
                / "failures.csv"
            )

            with failure_csv.open(
                "w",
                newline="",
                encoding="utf-8",
            ) as f:
                w = csv.writer(f)
                w.writerow(
                    [
                        "class",
                        "video",
                        "error",
                    ]
                )
                w.writerows(failures)

            print(
                "⚠️ Fallos de esta sesión:",
                failure_csv,
            )


🔄 ESTADO DE LOS 4 DATASETS
   datasets: 2
   dvs640_t0.1: 27/255 videos
   dvs640_t0.2: 27/255 videos
--------------------------------------------------------------------------------
   total pares H5+MP4: 54/510
   pendientes: 456
   progreso global: 10.59%
   checkpoint: /content/drive/MyDrive/DATASET_GEN/progress.json
🚀 INICIANDO / REANUDANDO LOS 4 DATASETS
   configuraciones ya listas: 54
   pendientes: 456
⏭️ [1/255] brush_hair/Brushing_Her_Hair__[_NEW_AUDIO_]_UPDATED!!!!_brush_hair_h_cm_np1_le_goo_3.avi: 4/4 datasets completos
⏭️ [2/255] brush_hair/Brushing_my_Long_Hair__February_2009_brush_hair_u_nm_np1_ba_goo_1.avi: 4/4 datasets completos
⏭️ [3/255] brush_hair/Ella_brushing_her_amazing_long_hair_brush_hair_u_cm_np1_ba_goo_0.avi: 4/4 datasets completos
⏭️ [4/255] brush_hair/atempting_to_brush_my_hair_brush_hair_u_nm_np2_le_goo_0.avi: 4/4 datasets completos
⏭️ [5/255] brush_hair/brushing_hair_brush_hair_f_nm_np2_ba_goo_4.avi: 4/4 datasets completos
⏭️ [6/255] cartwheel/Cartwheel_

## Estructura final de los 4 datasets

```text
HMDB51_V2E/
│
├── dvs128_t0.1/
│   ├── brush_hair/
│   │   ├── nombre_video_1.h5
│   │   ├── nombre_video_1.mp4
│   │   ├── nombre_video_2.h5
│   │   ├── nombre_video_2.mp4
│   │   └── ... hasta 5 videos
│   ├── cartwheel/
│   ├── catch/
│   └── ... hasta completar 51 clases
│
├── dvs128_t0.2/
│   └── mismas 51 clases × mismos 5 videos
│
├── dvs128_t0.3/
│   └── mismas 51 clases × mismos 5 videos
│
├── dvs640_t0.3/
│   └── mismas 51 clases × mismos 5 videos
│
├── selection_manifest.csv
├── processing_manifest.csv
└── progress.json
```

### Totales

| Dataset | Cámara | Threshold | Clases | Videos |
|---|---|---:|---:|---:|
| `dvs128_t0.1` | dvs128 | 0.1 | 51 | 255 |
| `dvs128_t0.2` | dvs128 | 0.2 | 51 | 255 |
| `dvs128_t0.3` | dvs128 | 0.3 | 51 | 255 |
| `dvs640_t0.3` | dvs640 | 0.3 | 51 | 255 |

Cada video conserva su nombre original y genera un par `.h5 + .mp4`.